In [ ]:
from typing import TypedDict,Annotated
from operator import add
from langgraph.graph import StateGraph,START,END
from langgraph.types import Overwrite


# 1. 定义状态 / Reducer
class OverAllState(TypedDict):
    # 归约的方式是add追加合并
    logs:Annotated[list[str],add]
    cur_id: str


# 2. 定义节点
def node_1(state:OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"k:{k} v:{v}")
    return {
        "logs":["node_1 运行完毕"]
    }

def node_2(state:OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"k:{k} v:{v}")
    return {
        "logs":Overwrite(["node_2 运行完毕"]) # Overwrite绕过Reducer
        # 会把之前所有的内容直接覆盖，因为使用了override,也就是["node_1 运行完毕"]被覆盖了
    }

def node_3(state:OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"k:{k} v:{v}")
    return {
        "logs":["node_3 运行完毕"]
    }


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1",node_1)
builder.add_node("node_2",node_2) 
builder.add_node("node_3",node_3)
builder.add_edge(START,"node_1")
builder.add_edge("node_1","node_2")
builder.add_edge("node_2","node_3")
builder.add_edge("node_3",END)

graph = builder.compile()
result = graph.invoke({"logs":["start"],"cur_id":"start"})
print(result)


k:logs v:['start']
k:cur_id v:start
k:logs v:['start', 'node_1 运行完毕']
k:cur_id v:start
k:logs v:['node_2 运行完毕']
k:cur_id v:start
{'logs': ['node_2 运行完毕', 'node_3 运行完毕'], 'cur_id': 'start'}
